# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"Dataset: {getattr(dataset.metadata, 'name', None)}\nDescription: {getattr(dataset.metadata, 'description', None)}\n")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and columns. All entities are referenced by their `@id` fields for clarity and reproducibility.

Let's print a summary of record sets and their fields (by `@id`).

In [ ]:
# Get a summary of all record sets, their @ids, and the field @ids they contain
record_sets = getattr(dataset.metadata, 'record_sets', [])

if not record_sets:
    # Some Croissant exports may use 'recordSet' instead due to schema variations
    record_sets = getattr(dataset.metadata, 'recordSet', [])

print(f"Number of record sets: {len(record_sets)}\n")

overview = []
for rset in record_sets:
    rec_id = getattr(rset, '@id', None)
    rec_name = getattr(rset, 'name', None)
    fields = getattr(rset, 'fields', [])
    if not fields:
        # Croissant may export as 'field' instead
        fields = getattr(rset, 'field', [])

    field_ids = []
    for f in fields:
        fid = getattr(f, '@id', str(f))
        fname = getattr(f, 'name', None)
        ftype = getattr(f, 'data_type', None)
        field_ids.append(fid)
    print(f"Record set @id: {rec_id}, name: {rec_name}\n  Fields @id: {field_ids}\n")
    overview.append({'@id': rec_id, 'name': rec_name, 'fields': field_ids})

if len(record_sets) == 0:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities are referenced by their `@id` fields, as recommended.

Below, all available record set `@id`s are loaded into DataFrames for inspection.

In [ ]:
# List all available record set @ids to load
record_set_ids = [rset['@id'] for rset in overview]
print("Available record set @ids:", record_set_ids)

dataframes = {}
for rec_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} records from record set @id: {rec_id}")
    except Exception as e:
        print(f"Could not load records from {rec_id}: {e}")

# Let's display columns for the first record set loaded
if dataframes:
    default_rec_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set @id: {default_rec_id}\n{dataframes[default_rec_id].columns.tolist()}")
    display(dataframes[default_rec_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps: filter, normalize, and group.

Reference all columns by their canonical `@id` (as columns in the DataFrame).

Let's select a numeric field and perform: filtering, normalization, and grouping.

In [ ]:
# View the available field (column) @ids
if not dataframes:
    print("No dataframes available for EDA.")
else:
    record_set_id = default_rec_id  # Use the first loaded record set
    df = dataframes[record_set_id]
    print("Available columns (field @ids):\n", df.columns.tolist())
    
    # Heuristically pick a numeric field (e.g. MW, coverage, abundance, etc.)
    candidate_numeric_fields = [col for col in df.columns 
                                if df[col].dtype in ['float64', 'int64'] or any(s in str(col).lower() for s in ['abundance', 'count', 'mw', 'coverage', 'area', 'score', 'peptide'])]
    numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else None
    print(f"Example numeric field for EDA: {numeric_field}")

    # Filtering
    threshold = df[numeric_field].mean() if numeric_field else None
    if numeric_field and threshold is not None:
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a good group field (heuristic: fields with < 10 unique values and dtype 'object')
        group_field = None
        for col in df.columns:
            if df[col].dtype == 'object' and df[col].nunique() < 10:
                if col != numeric_field:
                    group_field = col
                    break
        print(f"Grouping field selected: {group_field}")

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field and its relationship to a group field, if present.

For this example, a histogram and, if possible, a boxplot by group are shown.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No data loaded; cannot visualize.")
else:
    # Using prior selection
    if numeric_field is not None:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()
        
        if group_field is not None:
            plt.figure(figsize=(10,5))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f'{numeric_field} by {group_field}')
            plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load a Croissant-spec dataset using the `mlcroissant` library.

- We listed all available record sets and fields using their `@id` for reproducible reference.
- We loaded records from a chosen record set and performed basic exploratory data analysis, including filtering, normalization, and grouping by attributes.
- Data distributions and group differences were visualized to highlight trends in the dataset.

**Note:** Always reference record sets, fields, and columns by their unique `@id` for clarity and reproducibility in your analyses.